# 02 — Four-mall gravity / PPML models

**What we are doing:** Estimate whether driving distance predicts allocated device activity after controlling for the destination, month, income, inequality, population, and neighborhood composition.

$$E[Y_{ijt}|X]=\exp(\alpha+\beta_d\ln(1+d_{ij})+\gamma_j+\tau_t+\theta X_i)$$

**Why PPML?** It models nonnegative outcomes without taking the log of the dependent variable. It is **not** a causal design.

In [ ]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PROCESSED=ROOT/'data'/'processed'
FORMULAS={
'M1_distance':'flow ~ log_distance + C(Mall) + C(Month)',
'M2_income_population':'flow ~ log_distance + log_income + log_population + C(Mall) + C(Month)',
'M3_gini':'flow ~ log_distance + log_income + gini + log_population + C(Mall) + C(Month)',
'M4_race':'flow ~ log_distance + log_income + gini + black_share + hispanic_share + log_population + C(Mall) + C(Month)',
'M5_buckhead_income':'flow ~ log_distance + log_income + log_income:Buckhead + log_population + C(Mall) + C(Month)',
'M6_buckhead_black_share':'flow ~ log_distance + log_income + black_share + black_share:Buckhead + log_population + C(Mall) + C(Month)',
'M7_origin_FE':'flow ~ log_distance + C(ZCTA) + C(Mall) + C(Month)',
'M8_origin_FE_race_interaction':'flow ~ log_distance + black_share:Buckhead + C(ZCTA) + C(Mall) + C(Month)'}
results=[]
for scenario in ['2020_geometry_provisional','2010_equal_block_diagnostic']:
    panel=pd.read_csv(PROCESSED/f'panel_{scenario}.csv',dtype={'ZCTA':str})
    for name,formula in FORMULAS.items():
        model=smf.glm(formula,data=panel,family=sm.families.Poisson())
        groups=panel.loc[model.data.row_labels,'ZCTA']
        fit=model.fit(cov_type='cluster',cov_kwds={'groups':groups},maxiter=150)
        for term in fit.params.index:
            results.append(dict(scenario=scenario,model=name,term=term,coef=fit.params[term],se=fit.bse[term],p=fit.pvalues[term],n=fit.nobs))
output=pd.DataFrame(results)
print(output[output.term=='log_distance'][['scenario','model','coef','se','p','n']].to_string(index=False))
output.to_csv(PROCESSED/'rerun_four_mall_coefficients.csv',index=False)

## Interpret coefficients
**What we are doing:** Convert the distance coefficient into a comparison between two actual distances. Example: the predicted count ratio for 10 vs 5 driving miles is $\exp[\beta_d(\ln 11-\ln 6)]$. This is **not** a causal estimate.

In [ ]:
import numpy as np
beta=output.query("scenario=='2020_geometry_provisional' and model=='M2_income_population' and term=='log_distance'").coef.iloc[0]
ratio=np.exp(beta*(np.log1p(10)-np.log1p(5)))
print(f'Illustrative predicted activity ratio at 10 vs 5 miles: {ratio:.3f}')